# TP1 — Análisis Exploratorio y Preparación de Datos
### Inteligencia Artificial y Aprendizaje Automático I · 2026
### Dataset: transacciones inmobiliarias — Prefectura de Tokio (MLIT)

**Fuente:** MLIT de Japón, *Real Estate Transaction Price Information* (dato público). Trabajamos sobre
**toda la Prefectura de Tokio** (~340.000 registros × 27 variables), cubriendo los 23 wards especiales y
municipios. El **ward** (distrito) es la feature de ubicación principal: permite comparar precios entre
zonas y es, a priori, el predictor de precio más fuerte para el TP2.

Objetivo del TP1: **no predecir**, sino entender el dominio, detectar las imperfecciones reales del
dataset y dejarlo limpio + con features para el TP2 (regresión de precio) y el TP3 (clasificación de
estructura).

## 0. Setup

In [ ]:
import os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_columns", 30)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

RAW_DATA_PATH = r"C:\Users\camij\Downloads\Tokyo_20053_20253_en\Tokyo_20053_20253_en.csv"   # es un CSV con encoding Windows-1252
PROCESSED_DIR = "outputs/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

## A. Contexto y preguntas

### A.1 Dominio
Cada fila es **una operación de compraventa** reportada al MLIT entre 2005 y 2025, en cualquier ward o
municipio de la Prefectura de Tokio. La cobertura amplia incluye tanto zonas céntricas
comerciales/corporativas (ej. Chiyoda, Minato) como wards más residenciales, lo que da un dataset
heterogéneo donde la **ubicación (ward)** explica buena parte de la variación de precio.

### A.2 Columnas que requieren decisión (el resto son features directas o descartes obvios)
| Columna | Rol / problema |
|---|---|
| `Type` | Land and Building vs. Land Only → explica los nulos estructurales |
| `Total transaction value` | **TARGET TP2** (¥); cola derecha extrema → `log1p` |
| `Building : Structure` | **TARGET TP3** (W/RC/SRC/S); nula en Land Only |
| `City,Town,Ward,Village` | **Ward** → feature de ubicación principal (One-Hot) |
| `Area(㎡)`, `Building : Total floor area` | Artefacto `‡u` + tope censurado `"2000 or greater"` |
| `Building : Construction year` | Texto (incluye `"before the war"`) → derivar antigüedad |
| `District`, `Nearest station : Name` | Alta cardinalidad → agrupar + frequency encoding |
| `Land : Price per m2` | Nulo estructural (solo Land Only) |
| `Prefecture`, `City code` | Constantes/redundantes → descartar (detección automática) |
| `Transaction factors` | Mayormente nulo → flag binario |

### A.3 Diez preguntas de interés
**Descriptivas:** (1) distribución del precio y peso de la cola derecha; (2) precio/m² Comercial vs.
Residencial; (3) estructuras predominantes y su relación con el año; (4) distancia a estación vs.
precio; (5) evolución de volumen y precio 2005–2025; (6) proporción Land Only vs. Land and Building y
su efecto en los nulos; (7) **qué wards concentran más operaciones y mayores precios** (comparación
geográfica que habilita usar toda la prefectura).
**Predicción (TP2):** (8) estimar `Total transaction value` con el resto de variables (incluido el ward).
**Clasificación (TP3):** (9) clasificar `Building : Structure`; (10) si zona+tamaño+antigüedad+ward
separan las clases o se solapan (RC vs. SRC).

## 1. Carga de datos
El `.xls` del MLIT es en realidad un **CSV en Windows-1252**. Con la codificación por defecto, el
carácter ㎡ se corrompe a `‡u` (uno de los problemas de calidad que resolvemos abajo). Cargamos **toda
la Prefectura de Tokio** (~340.000 filas) y derivamos la columna `ward` como feature de ubicación.

In [ ]:
df = pd.read_csv(RAW_DATA_PATH, encoding="cp1252", low_memory=False).reset_index(drop=True)
print("Shape del dataset completo (toda la Prefectura de Tokio):", df.shape)

# Ward como feature de ubicación principal: normalizamos el nombre (quitamos el sufijo " Ward"/" City").
df["ward"] = (df["City,Town,Ward,Village"].astype(str)
              .str.replace(r"\s+(Ward|City|Town|Village)$", "", regex=True)
              .str.strip())
print("Cantidad de wards/municipios distintos:", df["ward"].nunique())
df["ward"].value_counts().head(10)

In [ ]:
df.info()


## 2. Vista general y calidad de datos

### 2.1 El dataset mezcla dos naturalezas
`Type` define dos universos: **Land and Building** (tiene edificio → estructura, año y superficie
construida) y **Land Only** (esas columnas **no existen conceptualmente** → nulo estructural, no error).
Esto explica gran parte de los nulos y condiciona el tratamiento columna por columna.

In [ ]:
df["Type"].value_counts()


In [ ]:
nulls = (df.isna().mean() * 100).round(2).sort_values(ascending=False)
nulls = nulls[nulls > 0]
nulls.to_frame("% nulos")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
sns.barplot(x=nulls.values, y=nulls.index, ax=ax, color="#4C72B0")
ax.set_xlabel("% de valores nulos")
ax.set_ylabel("")
ax.set_title("Proporción de valores faltantes por columna — Prefectura de Tokio")
plt.tight_layout()
plt.show()


In [ ]:
# ¿Los nulos de las columnas "de edificio" están explicados por Type?
cols_edificio = ["Building : Structure", "Building : Total floor area", "Building : Construction year"]
tabla_nulos_por_tipo = df.groupby("Type")[cols_edificio].apply(lambda g: g.isna().mean().round(3) * 100)
tabla_nulos_por_tipo


**Lectura:** en `Land Only` las columnas de edificio son ~100% nulas (nulo estructural). Dentro de
`Land and Building`, `Building : Structure` conserva un pequeño % de nulos genuinos (sí faltantes
reales). `Land : Price per m2` es la contracara: solo existe en `Land Only`.

In [ ]:
df.groupby("Type")["Land : Price per m2"].apply(lambda s: s.isna().mean().round(3) * 100)


## 3. Limpieza y tratamiento de calidad de datos
Documentamos columna por columna el problema y la transformación (insumo directo del criterio
"Ingeniería de Datos" de la rúbrica y del paper del TP4).

### 3.1 Reto 1 — Artefacto de encoding en superficies (`‡u`)
`Area(㎡)` y `Building : Total floor area` traen un valor censurado (`"2,000 ‡u or greater."`) en los
lotes/edificios más grandes. Un `pd.to_numeric` directo los volvería `NaN` y **perdería el tramo más
grande/caro** — el peor lugar para perder datos.

In [ ]:
col_area_raw = "Area(‡u)"
col_floor_raw = "Building : Total floor area"

print("Casos censurados en Area:", df[col_area_raw].astype(str).str.contains("greater", na=False).sum())
print("Casos censurados en Building total floor area:", df[col_floor_raw].astype(str).str.contains("greater", na=False).sum())
df[[col_area_raw, col_floor_raw]].sample(5, random_state=RANDOM_STATE)


In [ ]:
def limpiar_superficie(serie, tope=2000):
    """Extrae el valor numérico de una columna de superficie del MLIT.
    Devuelve (valores_limpios, flag_capado) donde el flag indica que el valor
    real es >= tope (censurado por el propio dataset, no un outlier de carga)."""
    s = serie.astype(str).str.replace(",", "", regex=False)
    es_capado = s.str.contains("greater", na=False)
    numeros = s.str.extract(r"(\d+\.?\d*)")[0]
    valores = pd.to_numeric(numeros, errors="coerce")
    valores = valores.where(~es_capado, tope)
    return valores, es_capado.fillna(False)

df["area_terreno_m2"], df["area_terreno_capada"] = limpiar_superficie(df[col_area_raw])
df["area_construida_m2"], df["area_construida_capada"] = limpiar_superficie(df[col_floor_raw])

print("Filas con área de terreno capada a 2000 m²:", df["area_terreno_capada"].sum())
print("Filas con área construida capada a 2000 m²:", df["area_construida_capada"].sum())
df[[col_area_raw, "area_terreno_m2", "area_terreno_capada",
    col_floor_raw, "area_construida_m2", "area_construida_capada"]].sample(6, random_state=RANDOM_STATE)


**Decisión:** extraer el número con regex, quitar comas de millar y tratar `"2,000 or greater"`
como un tope censurado — capeamos el valor a 2000 y creamos un flag (`area_terreno_capada` /
`area_construida_capada`) para que los modelos posteriores puedan distinguir "2000 real" de "2000 censurado".
Nunca eliminamos estas filas: son, justamente, las operaciones más grandes de Chiyoda (zona de oficinas).


### 3.2.b Reto extra — `Nearest station : Distance` no es numérica
La columna llega como **texto**: además de minutos numéricos trae **rangos** (`"30-60minutes"`,
`"1H-1H30"`, `"2H-"`). Un `pd.to_numeric` directo la convierte en `NaN` (o rompe el escalado), y es una
de las features más fuertes para el precio.

**Decisión:** parsear los rangos a un valor numérico en minutos (punto medio del intervalo; para los
abiertos tipo `"2H-"`, el extremo inferior), dejando un flag de los casos que no se pudieron convertir.

In [ ]:
def convertir_distancia_estacion(serie):
    """Convierte 'Nearest station : Distance' a minutos numéricos.

    Formatos contemplados:
      - numérico puro: "15"            -> 15
      - rango en minutos: "30-60minutes" -> 45 (punto medio)
      - rango en horas: "1H-1H30"      -> 75 (punto medio)
      - abierto: "2H-"                 -> 120 (extremo inferior)
    """
    s = serie.astype(str).str.strip()

    # 1) Los que ya son numéricos.
    resultado = pd.to_numeric(s, errors="coerce")

    def a_minutos(txt):
        """Convierte un token tipo '1H30', '2H', '45' a minutos."""
        m = re.fullmatch(r"(\d+)H(\d+)?", txt)
        if m:
            return int(m.group(1)) * 60 + (int(m.group(2)) if m.group(2) else 0)
        m = re.fullmatch(r"(\d+)", txt)
        return int(m.group(1)) if m else None

    def parsear(txt):
        t = txt.replace("minutes", "").replace("minute", "").strip()
        if not t or t.lower() == "nan":
            return np.nan
        partes = [p for p in t.split("-")]
        valores = [a_minutos(p) for p in partes if p]
        valores = [v for v in valores if v is not None]
        if not valores:
            return np.nan
        if len(valores) == 1:
            return float(valores[0])          # abierto ("2H-") o valor único
        return float(np.mean(valores[:2]))    # punto medio del rango

    pendientes = resultado.isna() & s.notna()
    resultado[pendientes] = s[pendientes].apply(parsear)
    return pd.to_numeric(resultado, errors="coerce")


df["distancia_estacion_min"] = convertir_distancia_estacion(df["Nearest station : Distance"])

# Diagnóstico: qué valores de texto no se pudieron convertir (deberían ser solo los nulos originales).
no_convertidos = df.loc[df["distancia_estacion_min"].isna() & df["Nearest station : Distance"].notna(),
                        "Nearest station : Distance"]
print("Valores no convertidos:", no_convertidos.nunique())
if no_convertidos.nunique() > 0:
    print(no_convertidos.value_counts().head(10))

df["distancia_estacion_nula"] = df["distancia_estacion_min"].isna().astype(int)
print("\nDistribución de la distancia a estación (minutos):")
print(df["distancia_estacion_min"].describe().round(2))

### 3.2 Reto 2 — Nulos estructurales por `Type` (no son errores)

Ya vimos que en `Land Only` las columnas de edificio no aplican conceptualmente. Imputar con media/mediana
inventaría un edificio que no existe. En cambio, creamos un indicador explícito y un relleno neutro.


In [ ]:
df["sin_edificio"] = (df["Type"] == "Residential Land(Land Only)").astype(int)

# Building : Total floor area -> 0 tiene sentido semántico (no hay construcción)
df["area_construida_m2"] = df["area_construida_m2"].where(df["sin_edificio"] == 0, 0.0)

# Building : Construction year -> no imputamos; el flag sin_edificio ya explica el nulo.
# "before the war" es una categoría temporal real (construcciones anteriores a 1926) -> la
# tratamos aparte en vez de descartarla o forzarla a NaN.
df["antes_de_guerra"] = (df["Building : Construction year"] == "before the war").astype(int)
anio_construccion = pd.to_numeric(df["Building : Construction year"], errors="coerce")
df["anio_construccion"] = anio_construccion

print("Filas marcadas antes_de_guerra:", df["antes_de_guerra"].sum())
df[["Type", "sin_edificio", "Building : Construction year", "anio_construccion", "antes_de_guerra"]].query("antes_de_guerra == 1")


In [ ]:
# Land : Price per m2 -> lo dejamos tal cual, sin imputar: solo tiene sentido para Land Only,
# y ya está disponible ahí en (casi) el 100% de los casos.
df["precio_suelo_m2"] = df["Land : Price per m2"]

# Building : Structure (target del TP3) -> NO se imputa: los NaN genuinos dentro de
# "Land and Building" se dejan como faltantes reales, y las filas de "Land Only" simplemente
# no participan del problema de clasificación del TP3 (no tienen estructura por definición).
print(df.groupby("Type")["Building : Structure"].apply(lambda s: s.isna().sum()))


### 3.4.b Reto extra — el target del TP3 es multi-etiqueta
`Building : Structure` **no tiene 4 categorías limpias**. Además de `W`, `S`, `RC` y `SRC` aparecen:

- clases minoritarias adicionales (`LS` = *light steel*, `B` = *block*),
- y sobre todo **combinaciones multi-valor** (`"RC, W"`, `"S, W"`, `"RC, S, W, B"`), que corresponden a
  edificios con más de un sistema constructivo.

Tomado tal cual, el target tendría ~25 categorías, muchas con un puñado de casos: el TP3 entrenaría
sobre clases imposibles de aprender y las métricas macro se volverían ruido.

**Decisión:** derivar `estructura_principal` quedándose con el **primer sistema constructivo listado**
(el principal) y agrupando en `"Otros"` todo lo que no sea `W`/`S`/`RC`/`SRC`. Se documenta qué
porcentaje de filas era multi-etiqueta, porque es una simplificación que hay que declarar en el paper.

In [ ]:
# El target del TP3 llega como texto multi-valor ("RC, W") y con clases raras (LS, B).
CLASES_PRINCIPALES = ["W", "S", "RC", "SRC"]

pct_multi = df["Building : Structure"].str.contains(",", na=False).mean() * 100
print(f"Filas con estructura multi-valor: {pct_multi:.2f}%")
print(f"Categorías crudas en Building : Structure: {df['Building : Structure'].nunique()}")

# Nos quedamos con el sistema constructivo principal (el primero listado).
estructura_principal = df["Building : Structure"].str.split(",").str[0].str.strip()

# Todo lo que no sea una de las 4 clases principales -> "Otros" (LS, B, etc.).
df["estructura_principal"] = estructura_principal.where(
    estructura_principal.isin(CLASES_PRINCIPALES), "Otros")

# Preservamos el nulo estructural: Land Only no tiene edificio.
df.loc[df["Building : Structure"].isna(), "estructura_principal"] = np.nan

print("\nDistribución del target normalizado (estructura_principal):")
print(df["estructura_principal"].value_counts(dropna=False))
print("\nProporción (%):")
print((df["estructura_principal"].value_counts(normalize=True) * 100).round(2))

**Decisión:** separamos el tratamiento por `Type`. Para las columnas de edificio en `Land Only`
usamos un indicador (`sin_edificio=1`) + relleno neutro (0 en superficie construida, nulo explícito en
año/estructura), en vez de imputación estadística. El pequeño remanente de nulos genuinos de
`Building : Structure` dentro de `Land and Building` (~4.4%) se deja como faltante real —no se imputa
la variable *target* del TP3—, y esas filas simplemente se excluirán del entrenamiento de clasificación.


### 3.3 Reto 3 — Cardinalidad geográfica (`ward`, `District`, estación)
Con toda la prefectura hay tres niveles de ubicación anidados: **ward** (59 categorías), **District**
(~1.450) y **estación** (~660). Estrategia: `ward` va a **One-Hot** (interpretable, es el predictor de
ubicación más fuerte); `District` y estación se agrupan por **top-N** (las 100 más frecuentes, el resto
a `"Otros"`) y se codifican por **frecuencia**.

> **Por qué top-N y no un umbral porcentual:** con 344k filas, un umbral de 0.5% exige ~1.720 casos por
> categoría y dejaría el 98% de las filas en `"Otros"`, destruyendo la señal geográfica. Top-N controla
> el número de categorías con independencia del tamaño del dataset.

In [ ]:
print("Categorías únicas en ward:", df["ward"].nunique())
print("Categorías únicas en District:", df["District"].nunique())
print("Categorías únicas en Nearest station:", df["Nearest station : Name"].nunique())
df["District"].value_counts().head(10)

In [ ]:
def agrupar_top_n(serie, n=100, etiqueta="Otros"):
    """Conserva las N categorías más frecuentes; el resto se agrupa en 'Otros'.

    Con ~344k filas y >1.400 barrios, un umbral porcentual (ej. 0.5%) exigiría miles de casos por
    categoría y mandaría casi todo a 'Otros', destruyendo la señal. Top-N controla directamente
    cuántas columnas/categorías generamos, independientemente del tamaño del dataset.
    """
    top = serie.value_counts().nlargest(n).index
    return serie.where(serie.isin(top), etiqueta)

In [ ]:
# Top-N: conservamos las 100 categorías más frecuentes de cada variable geográfica.
TOP_N_GEO = 100

df["district_agrupado"] = agrupar_top_n(df["District"], n=TOP_N_GEO)
df["estacion_agrupada"] = agrupar_top_n(df["Nearest station : Name"], n=TOP_N_GEO)

for col_orig, col_agr in [("District", "district_agrupado"),
                          ("Nearest station : Name", "estacion_agrupada")]:
    cobertura = (df[col_agr] != "Otros").mean() * 100
    print(f"{col_orig}: {df[col_orig].nunique()} categorías -> {df[col_agr].nunique()} "
          f"| filas fuera de 'Otros': {cobertura:.1f}%")

df["district_agrupado"].value_counts().head(10)

**Decisión:** `ward` se codifica con One-Hot completo (59 columnas, no se agrupa). `District` y estación
se reducen a las 100 categorías más frecuentes + `"Otros"` y se aplican por frequency encoding.
`distancia_estacion_min` se mantiene numérica: captura parte del efecto ubicación sin sumar columnas.

### 3.4 Otros problemas de calidad detectados
- **Constantes**: se detectan automáticamente las columnas con un solo valor (ej. `Prefecture` = Tokyo)
  y se descartan por no aportar varianza. `City,Town,Ward,Village` **ya no es constante** (varía por
  ward): se conserva como feature `ward`; su `code` se descarta por redundante.
- `Price information classification`: baja variabilidad → se descarta.
- `Transaction factors`: mayormente nulo, pero "vacío vs. con nota" es informativo → flag binario.
- `Frontage`, `Frontage road : Width/Type/Direction`, `Land : Shape`: nulos puntuales (no
  estructurales) → imputación por mediana/moda dentro de cada zona.

In [ ]:
# Detección automática de columnas constantes (sin varianza informativa en este dataset).
constantes = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
print("Columnas constantes detectadas (se descartan):", constantes)

print("\nDistribución de Price information classification:")
print(df["Price information classification"].value_counts(dropna=False))

df["tiene_nota_transaccion"] = df["Transaction factors"].notna().astype(int)

# Descartes: constantes detectadas + la columna cruda de ward (ya representada por 'ward')
# + code de ward (redundante con 'ward') + clasificación de precio (poca info).
a_descartar = set(constantes) | {
    "City,Town,Ward,Village", "City,Town,Ward,Village code",
    "Price information classification",
}
df = df.drop(columns=[c for c in a_descartar if c in df.columns])
print("\nShape luego de descartar constantes/redundantes:", df.shape)

In [ ]:
# Imputación puntual (no estructural) por mediana/moda dentro de cada zona (Area: Comercial/Residencial).
# OJO: 'Area' tiene nulos propios, y groupby descarta ese grupo -> hace falta un fallback global,
# si no quedan filas sin imputar (las de 'Contract Price Information', que no traen zona).
for col in ["Frontage", "Frontage road : Width"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = df.groupby("Area")[col].transform(lambda s: s.fillna(s.median()))
    df[col] = df[col].fillna(df[col].median())          # fallback global

for col in ["Frontage road : Type", "Frontage road : Direction", "Land : Shape"]:
    moda = df[col].mode(dropna=True)[0]
    df[col] = df[col].fillna(moda)

print("Nulos restantes tras imputar:")
print(df[["Frontage", "Frontage road : Width", "Frontage road : Type",
          "Frontage road : Direction", "Land : Shape"]].isna().sum())

## 4. Ingeniería de características

### 4.1 Variables temporales

`Transaction timing` llega como texto (`"2nd quarter 2025"`). Lo separamos en año y trimestre numéricos,
y derivamos la **antigüedad** del edificio al momento de la operación.


In [ ]:
extraido = df["Transaction timing"].str.extract(r"(?P<ord>\d(?:st|nd|rd|th)) quarter (?P<anio>\d{4})")
df["anio_operacion"] = pd.to_numeric(extraido["anio"], errors="coerce")
df["trimestre_operacion"] = extraido["ord"].str[0].astype(int)
df["antiguedad"] = df["anio_operacion"] - df["anio_construccion"]

# Control de calidad: una antigüedad negativa significa que el edificio figura construido DESPUÉS de
# la operación -> inconsistencia de carga del MLIT. No se puede dejar pasar (rompe la interpretación
# de la variable y ensucia la correlación con el precio).
# Decisión: marcar con un flag y llevar a 0 (la operación es esencialmente de obra nueva).
neg = (df["antiguedad"] < 0)
df["antiguedad_inconsistente"] = neg.astype(int)
df.loc[neg, "antiguedad"] = 0
print(f"Antigüedades negativas detectadas y corregidas a 0: {neg.sum():,}")

df[["Transaction timing", "anio_operacion", "trimestre_operacion",
    "anio_construccion", "antiguedad", "antiguedad_inconsistente"]].sample(5, random_state=RANDOM_STATE)

### 4.2 Densidad constructiva y uso del suelo

In [ ]:
# Ratio construido: cuánto se construyó respecto a la superficie del terreno (solo tiene sentido
# cuando hay edificio; en Land Only queda en 0 por construcción, ya que area_construida_m2 = 0 ahí).
df["ratio_construido"] = df["area_construida_m2"] / df["area_terreno_m2"].replace(0, np.nan)

# Control de coherencia: 'Floor area ratio' es el coeficiente legal de edificabilidad (en %).
# Un ratio_construido muy por encima de ese límite sugiere superficie de terreno mal cargada.
limite_legal = df["Floor area ratio"] / 100
sospechosos = df["ratio_construido"] > (limite_legal * 1.5)
df["ratio_construido_sospechoso"] = sospechosos.fillna(False).astype(int)
print(f"Filas con ratio_construido incoherente con Floor area ratio: {df['ratio_construido_sospechoso'].sum():,}")
print("\nMáximo observado de ratio_construido:", round(df["ratio_construido"].max(), 2))

# Use es multi-valor (ej. "House, Office, Shop") -> lo separamos en flags binarios para los usos
# más frecuentes.
usos_relevantes = ["House", "Office", "Shop", "Warehouse", "Parking Lot", "Housing Complex"]
usos_normalizados = df["Use"].fillna("")
for uso in usos_relevantes:
    nombre_col = "uso_" + uso.lower().replace(" ", "_")
    df[nombre_col] = usos_normalizados.str.contains(uso, case=False, regex=False).astype(int)

df[["Use"] + ["uso_" + u.lower().replace(" ", "_") for u in usos_relevantes]].sample(6, random_state=RANDOM_STATE)

# Precio por m² de terreno: métrica normalizada, comparable entre inmuebles de distinto tamaño.
# NOTA: NO debe usarse como predictor en el TP2 (se deriva del target -> data leakage), pero es
# clave para el EDA y utilizable en el TP3 (donde el target es la estructura, no el precio).
df["precio_por_m2"] = df["Total transaction value"] / df["area_terreno_m2"].replace(0, np.nan)

### 4.3 Variables derivadas (resumen)
`area_terreno_m2` / `area_construida_m2` (+ flags `_capada`), `sin_edificio`, `antes_de_guerra`,
`anio_construccion`, `precio_suelo_m2`, `district_agrupado` / `estacion_agrupada`,
`tiene_nota_transaccion`, `anio_operacion` / `trimestre_operacion`, `antiguedad`, `ratio_construido`,
y 6 flags `uso_*` (split de `Use`).

## 5. Análisis exploratorio univariado

### 5.1 Variables numéricas

In [ ]:
numericas = ["Total transaction value", "precio_por_m2", "area_terreno_m2", "area_construida_m2",
             "distancia_estacion_min", "Frontage", "antiguedad", "ratio_construido"]
df[numericas].describe().T

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

sns.histplot(df["Total transaction value"], bins=50, ax=axes[0], color="#4C72B0")
axes[0].set_title("Precio total (¥) — escala original")

sns.histplot(np.log1p(df["Total transaction value"]), bins=50, ax=axes[1], color="#55A868")
axes[1].set_title("log1p(Precio total) — mucho más simétrico")

# NOTA: la última barra concentra todo lo recortado (>1000 m²), no es un pico real de la distribución.
sns.histplot(df["area_terreno_m2"].clip(upper=1000), bins=50, ax=axes[2], color="#C44E52")
axes[2].set_title("Superficie de terreno (m²)\n[recortado a 1000: la última barra acumula el resto]")

# Usamos la distancia YA PARSEADA a minutos (la columna original es texto y no se grafica bien).
sns.histplot(df["distancia_estacion_min"].dropna().clip(upper=60), bins=30, ax=axes[3], color="#8172B2")
axes[3].set_title("Distancia a la estación (min)\n[recortado a 60]")
axes[3].set_xlabel("distancia_estacion_min")

sns.histplot(df["antiguedad"].dropna(), bins=30, ax=axes[4], color="#CCB974")
axes[4].set_title("Antigüedad del edificio (años)")

sns.histplot(df["ratio_construido"].dropna().clip(upper=10), bins=40, ax=axes[5], color="#64B5CD")
axes[5].set_title("Ratio construido (superficie construida / terreno)")

plt.tight_layout()
plt.show()

**Lectura:** el precio en escala original está extremadamente sesgado a la derecha (operaciones
de miles de millones de yenes conviven con lotes chicos de pocos millones); `log1p` lo simetriza
notablemente. Esto anticipa la decisión que tomaremos en el TP2: modelar sobre `log1p(precio)`.


### 5.2 Variables categóricas

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.countplot(y=df["Type"], ax=axes[0, 0], order=df["Type"].value_counts().index, color="#4C72B0")
axes[0, 0].set_title("Type: terreno+edificio vs. solo terreno")

sns.countplot(y=df["Area"], ax=axes[0, 1], order=df["Area"].value_counts().index, color="#55A868")
axes[0, 1].set_title("Zona: Comercial vs. Residencial")

# Target YA NORMALIZADO (la columna cruda tiene ~25 categorías multi-valor ilegibles).
sns.countplot(y=df["estructura_principal"], ax=axes[1, 0],
              order=df["estructura_principal"].value_counts().index, color="#C44E52")
axes[1, 0].set_title("Estructura principal (target TP3, normalizado)")

# Top 10: con todas las categorías las barras chicas son invisibles.
top_city = df["City planning"].value_counts().head(10).index
sns.countplot(y=df["City planning"], ax=axes[1, 1], order=top_city, color="#8172B2")
axes[1, 1].set_title("Zonificación urbana (top 10)")

plt.tight_layout()
plt.show()

**Lectura:** el dataset es mayoritariamente **residencial**: `Residential Area` domina ampliamente sobre
`Commercial Area`, e `Industrial`/`Potential Residential` son marginales. En estructura, **`W` (madera)
es abrumadoramente mayoritaria** (~180k casos) frente a `S` y `RC` (~20k cada una), con `SRC` casi
testimonial. Esto invierte la intuición de un ward céntrico y define el TP3 como un problema de
**desbalance severo**: la accuracy será engañosa y F1 macro es obligatorio.

## 6. Análisis bivariado

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Con ~344k puntos el scatter se satura: submuestreamos para que se vea la nube real,
# y usamos escala logarítmica en el precio (la cola de ¥1e11 aplasta todo lo demás).
muestra = df.sample(min(30000, len(df)), random_state=RANDOM_STATE)

sns.scatterplot(data=muestra, x="area_terreno_m2", y="Total transaction value", hue="Type",
                alpha=0.3, s=12, edgecolor=None, ax=axes[0, 0])
axes[0, 0].set_yscale("log")
axes[0, 0].set_xlim(0, 1000)
axes[0, 0].set_title("Precio (escala log) vs. superficie de terreno")

sns.scatterplot(data=muestra, x="distancia_estacion_min", y="Total transaction value",
                alpha=0.3, s=12, edgecolor=None, ax=axes[0, 1], color="#55A868")
axes[0, 1].set_yscale("log")
axes[0, 1].set_xlim(0, 40)          # los pocos casos de 45-120 min estiran el eje inútilmente
axes[0, 1].set_title("Precio (escala log) vs. distancia a la estación (min)")

sns.boxplot(data=df, x="Area", y=np.log1p(df["Total transaction value"]), ax=axes[1, 0])
axes[1, 0].set_title("log1p(Precio) según zona")
axes[1, 0].tick_params(axis="x", rotation=20)

sns.boxplot(data=df, x="estructura_principal", y=np.log1p(df["Total transaction value"]),
            order=["W", "S", "RC", "SRC", "Otros"], ax=axes[1, 1])
axes[1, 1].set_title("log1p(Precio) según estructura principal")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="estructura_principal", y="anio_construccion",
            order=["W", "S", "RC", "SRC", "Otros"])
plt.title("Año de construcción según tipo de estructura")
plt.xlabel("Estructura principal")
plt.tight_layout()
plt.show()

### Matriz de correlación

In [ ]:
cols_corr = ["Total transaction value", "area_terreno_m2", "area_construida_m2",
             "distancia_estacion_min", "Frontage", "Frontage road : Width",
             "antiguedad", "ratio_construido", "Building coverage ratio", "Floor area ratio"]

corr = df[cols_corr].corr(numeric_only=True)

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Matriz de correlación — variables numéricas")
plt.tight_layout()
plt.show()

**Lectura:** las superficies son los predictores numéricos más fuertes del precio, pero con correlaciones
**moderadas**: `area_construida_m2` **0.34** y `area_terreno_m2` **0.30**.

Dos hallazgos que importan para el TP2:

- **`distancia_estacion_min` correlaciona apenas −0.07 con el precio**, mucho menos de lo esperado. Pero
  sí correlaciona fuerte con la zonificación: **−0.36 con `Building coverage ratio`** y **−0.33 con
  `Floor area ratio`**. Es decir, la cercanía a la estación captura sobre todo *densidad edificable
  permitida*, no precio directo. Relación contraintuitiva a discutir.
- **Colinealidad alta entre `Building coverage ratio` y `Floor area ratio` (0.81)** y entre
  `ratio_construido` y `area_construida_m2` (0.60): riesgo para el modelo lineal del TP2.

## 7. Detección y tratamiento de outliers

In [ ]:
def resumen_outliers_iqr(serie, k=1.5):
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    lim_inf, lim_sup = q1 - k * iqr, q3 + k * iqr
    outliers = serie[(serie < lim_inf) | (serie > lim_sup)]
    return {
        "limite_inferior": lim_inf, "limite_superior": lim_sup,
        "n_outliers": len(outliers), "pct_outliers": round(100 * len(outliers) / len(serie), 2),
    }

for col in ["Total transaction value", "area_terreno_m2"]:
    print(col, "->", resumen_outliers_iqr(df[col].dropna()))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Escala log: en escala lineal la caja se aplasta contra el eje por el outlier de ~¥1.7e11.
sns.boxplot(y=df["Total transaction value"], ax=axes[0], color="#4C72B0")
axes[0].set_yscale("log")
axes[0].set_title("Precio total (escala log) — outliers vía IQR")

sns.boxplot(y=df["area_terreno_m2"], ax=axes[1], color="#55A868")
axes[1].set_title("Superficie de terreno — outliers vía IQR")

plt.tight_layout()
plt.show()

**Decisión:** IQR marca ~10–15% como "outliers" en precio/superficie. **No los eliminamos**: son
operaciones reales de un distrito de oficinas de alto valor, no errores de carga. Estrategia: no
eliminar sino **transformar** — `log1p` en el precio y flags `_capada` para las superficies censuradas.
Se retoma en el cuestionario (pregunta 3).

## 8. Respuestas a las preguntas analíticas

**1. Distribución del precio y cola derecha.**

In [ ]:
mediana_precio = df["Total transaction value"].median()
media_precio = df["Total transaction value"].mean()
print(f"Mediana: ¥{mediana_precio:,.0f} — Media: ¥{media_precio:,.0f} — Máximo: ¥{df['Total transaction value'].max():,.0f}")
print(f"Asimetría (skew): {df['Total transaction value'].skew():.2f}")
print(f"Asimetría luego de log1p: {np.log1p(df['Total transaction value']).skew():.2f}")


Media ≫ mediana y skew alto en escala original confirman la cola pesada; `log1p` reduce fuertemente la
asimetría → justifica modelar el target en esa escala en el TP2.

**2. ¿Precio/m² difiere entre zona Comercial y Residencial?**

In [ ]:
# Comparación del precio por m² entre zonas (la métrica ya fue derivada en la sección 4).
df.groupby("Area")["precio_por_m2"].describe()[["count", "mean", "50%", "std"]].round(0)

Sí, y la diferencia es grande. Precio mediano por m²: **Comercial ~¥857k**, **Residencial ~¥412k**,
**Industrial ~¥264k** y **Potential Residential ~¥27k**. La zona comercial más que duplica a la
residencial y muestra mucha más dispersión.

**3. ¿Qué estructuras predominan y cómo se relacionan con el año de construcción?**
Predomina `W` (madera) por amplio margen, seguida de `S` y `RC`. El boxplot año~estructura confirma la
hipótesis: **`W` es la más reciente (mediana ~2010)**, mientras que `S`, `RC` y `SRC` tienen medianas
alrededor de **1990**. Coherente con la construcción residencial en madera de las últimas dos décadas
frente al parque de hormigón/acero más antiguo.

**4. ¿La distancia a la estación se asocia con el precio?**
**Débilmente: r = −0.07.** Contra lo esperado, la distancia no explica el precio por sí sola en toda la
prefectura; su efecto real aparece indirectamente vía zonificación (ver matriz de correlación).

**5. ¿Cómo evolucionó el volumen y precio medio de operaciones 2005-2025?**

In [ ]:
evolucion = df.groupby("anio_operacion").agg(
    n_operaciones=("Total transaction value", "size"),
    precio_medio=("Total transaction value", "mean"),
).reset_index()

fig, ax1 = plt.subplots(figsize=(11, 5))
ax2 = ax1.twinx()
ax1.bar(evolucion["anio_operacion"], evolucion["n_operaciones"], color="#8ecae6", alpha=0.6, label="N° operaciones")
ax2.plot(evolucion["anio_operacion"], evolucion["precio_medio"], color="#c1121f", marker="o", label="Precio medio")
ax1.set_ylabel("N° de operaciones")
ax2.set_ylabel("Precio medio (¥)")
ax1.set_xlabel("Año")
ax1.set_title("Volumen y precio medio de operaciones por año — Prefectura de Tokio")
plt.tight_layout()
plt.show()


Se observa una caída de volumen alrededor de 2008-2009 (crisis
financiera global) y nuevamente en 2020-2021 (pandemia), con recuperación posterior; el precio medio es
muy volátil año a año porque cada operación individual en Chiyoda puede ser un edificio corporativo de
gran magnitud (pocas transacciones, alta varianza).

**6. ¿Qué proporción son "solo terreno" vs. "terreno+edificio"?**


In [ ]:
df["Type"].value_counts(normalize=True).round(3) * 100


Aproximadamente un 78% son `Land and Building` y un 22% `Land Only` — de ahí se derivan
directamente los nulos estructurales tratados en la sección 3.2.

**7. ¿Qué barrios (District) concentran más operaciones y mayores precios?**


In [ ]:
# Comparación entre wards: es el análisis que habilita usar toda la prefectura (antes, con un solo
# ward, esta pregunta no existía). Ordenamos por precio mediano por m², que normaliza por tamaño.
resumen_ward = (df.groupby("ward")
                  .agg(operaciones=("Total transaction value", "count"),
                       precio_mediano=("Total transaction value", "median"),
                       precio_m2_mediano=("precio_por_m2", "median"))
                  .sort_values("precio_m2_mediano", ascending=False))

print(f"Wards/municipios analizados: {len(resumen_ward)}")
resumen_ward.head(10).round(0)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

top_precio = resumen_ward.head(15)
sns.barplot(x=top_precio["precio_m2_mediano"], y=top_precio.index, ax=axes[0], color="#C1121F")
axes[0].set_title("Top 15 wards por precio mediano por m²")
axes[0].set_xlabel("¥ / m²")
axes[0].set_ylabel("")

top_volumen = resumen_ward.sort_values("operaciones", ascending=False).head(15)
sns.barplot(x=top_volumen["operaciones"], y=top_volumen.index, ax=axes[1], color="#4C72B0")
axes[1].set_title("Top 15 wards por volumen de operaciones")
axes[1].set_xlabel("N° de operaciones")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

Con toda la prefectura esta pregunta se vuelve una **comparación entre wards**, y el resultado es el
hallazgo más claro del TP1:

- **Por precio/m²:** `Chiyoda` (~¥2,4M/m²), `Chuo` y `Minato` (~¥2,0M/m²) encabezan; en el otro extremo
  del top 15, `Sumida` y `Nakano` rondan los **¥0,63M/m²**. Es una diferencia de casi **4×** entre wards
  de la misma prefectura.
- **Por volumen:** el ranking es **completamente distinto**. `Setagaya`, `Nerima` y `Adachi` lideran en
  cantidad de operaciones (>18k cada uno), pero ninguno aparece arriba en precio; a la inversa,
  `Chiyoda` es el más caro pero con muy pocas transacciones.

Esa disociación **precio ↔ volumen** justifica empíricamente usar `ward` como feature de ubicación en el
TP2 y el TP3: aporta información que ni la superficie ni la distancia a estación capturan.

**8-10. Predicción y clasificación:** ver TP2 (regresión sobre `Total transaction value`) y TP3
(clasificación de `estructura_principal`) en la hoja de ruta del proyecto integrador.

## 9. Ingeniería de características: codificación y escalado

### 9.1 Codificación de variables categóricas

- **One-Hot Encoding** para categóricas de baja cardinalidad, donde no existe un orden natural:
  `Type`, `Area`, `Land : Shape`, `Frontage road : Direction`, `Frontage road : Type`, `City planning`.
- **Frequency encoding** para las categóricas de alta cardinalidad ya agrupadas: `district_agrupado`,
  `estacion_agrupada` (alternativa: One-Hot, ya viable tras el agrupamiento en "Otros").
- `Building : Structure` (target del TP3) **no se codifica acá**: se deja como etiqueta categórica para
  el TP3.


In [ ]:
# One-Hot para categóricas de baja/media cardinalidad. 'ward' se incluye acá: es la feature de
# ubicación principal y es interpretable (una columna por ward).
cols_onehot = ["ward", "Type", "Area", "Land : Shape", "Frontage road : Direction",
               "Frontage road : Type", "City planning"]

df_encoded = pd.get_dummies(df, columns=cols_onehot, prefix=cols_onehot)

# Alta cardinalidad -> frequency encoding (One-Hot explotaría la dimensionalidad).
for col in ["district_agrupado", "estacion_agrupada"]:
    frecuencias = df[col].value_counts(normalize=True)
    df_encoded[col + "_freq"] = df[col].map(frecuencias)

print("Shape luego de codificar:", df_encoded.shape)
print("Columnas de ward (One-Hot):", len([c for c in df_encoded.columns if c.startswith("ward_")]))
[c for c in df_encoded.columns if c.startswith("ward_")][:10]

### 9.2 ¿Es necesario escalar?
Depende del modelo (se retoma en TP2/TP3): **sí** para lineales regularizados, KNN y SVM (comparan
distancias / penalizan coeficientes en la misma escala); **no** para árboles (RF, XGBoost), invariantes
a la escala. Dejamos un `StandardScaler` de referencia sobre las numéricas continuas.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Usamos la distancia ya parseada a minutos (la columna original era texto y rompía el escalado).
cols_numericas_continuas = ["area_terreno_m2", "area_construida_m2", "distancia_estacion_min",
                            "Frontage", "Frontage road : Width", "antiguedad", "ratio_construido"]

scaler = StandardScaler()
ejemplo_escalado = scaler.fit_transform(df_encoded[cols_numericas_continuas].fillna(0))
pd.DataFrame(ejemplo_escalado, columns=cols_numericas_continuas).describe().loc[["mean", "std"]].round(2)

## 10. Cuestionario de Interpretación de Resultados

**1. ¿Cuáles variables parecen ser los predictores más fuertes y por qué? ¿Hubo alguna relación
contraintuitiva?**

Los predictores numéricos más fuertes son las **superficies**: `area_construida_m2` (r = 0.34) y
`area_terreno_m2` (r = 0.30). Pero el predictor más informativo no es numérico sino **geográfico**: el
`ward` separa precios medianos por m² en un rango de casi **4×** (Chiyoda ~¥2,4M vs. Sumida ~¥0,63M).

**Relación contraintuitiva:** esperábamos que la distancia a la estación fuera un predictor fuerte del
precio, y resultó **casi nula (r = −0.07)**. Su correlación relevante es con la zonificación
(−0.36 con `Building coverage ratio`, −0.33 con `Floor area ratio`): estar cerca de una estación implica
mayor densidad edificable permitida, y el efecto sobre el precio llega por esa vía indirecta, no
directamente. Además, gran parte de Tokio está bien conectada, lo que comprime la variabilidad útil.

**2. ¿Qué sesgos o limitaciones inherentes presenta el dataset?**

- **Dato autoreportado** por las partes de la operación (encuesta del MLIT): posible subreporte y
  redondeo de montos.
- **Cobertura limitada a la Prefectura de Tokio**: un modelo entrenado acá no generaliza a otras
  prefecturas.
- **Desbalance severo en el target del TP3**: `W` concentra la enorme mayoría de los casos frente a
  `S`/`RC`/`SRC`. Obliga a `class_weight`/SMOTE y a priorizar F1 macro sobre accuracy.
- **Simplificación declarada**: `Building : Structure` es multi-etiqueta (`"RC, W"`) y se redujo al
  sistema constructivo principal, agrupando `LS`/`B` en `"Otros"`.
- **Faltantes no aleatorios**: ~4,4% de las filas *Land and Building* no reportan estructura; si ese
  faltante se concentra en cierto tipo de edificio, el clasificador del TP3 aprende sobre una muestra
  sesgada.

**3. ¿Los outliers de precio/superficie son errores o fenómenos genuinos? ¿Cómo impacta la decisión de
predecir o ignorarlos?**

Son mayormente **fenómenos genuinos**: el IQR marca ~11,2% del precio y ~10,4% de la superficie como
outliers, pero Tokio realmente contiene desde lotes residenciales chicos hasta operaciones corporativas
de miles de millones de ¥ (máximo observado: **¥170.000M**). No son errores de carga como sí lo eran las
**antigüedades negativas** (corregidas explícitamente).

Eliminarlos sesgaría el modelo hacia el segmento chico y le impediría estimar las operaciones de mayor
valor. Por eso la estrategia es **conservarlos y transformar la escala**: el skew del precio pasa de
**150,4 a 0,38 con `log1p`**, lo que justifica modelar el target en esa escala en el TP2.

## 11. Dataset limpio — insumo para TP2 y TP3
Serializamos el dataset transformado para reutilizarlo sin recalcular.

In [ ]:
columnas_finales = [
    # identificación / contexto
    "ward",
    "District", "district_agrupado", "Nearest station : Name", "estacion_agrupada",
    "Type", "Area", "City planning", "Land : Shape",
    "Frontage road : Direction", "Frontage road : Type",
    # temporales
    "anio_operacion", "trimestre_operacion", "anio_construccion", "antiguedad", "antes_de_guerra",
    "antiguedad_inconsistente",
    # superficie y forma
    "area_terreno_m2", "area_terreno_capada", "area_construida_m2", "area_construida_capada",
    "ratio_construido", "ratio_construido_sospechoso", "Frontage", "Frontage road : Width",
    "Building coverage ratio", "Floor area ratio",
    # ubicación
    "distancia_estacion_min", "distancia_estacion_nula",
    # usos
    "uso_house", "uso_office", "uso_shop", "uso_warehouse", "uso_parking_lot", "uso_housing_complex",
    # flags
    "sin_edificio", "tiene_nota_transaccion",
    # precio del suelo (solo Land Only) y precio normalizado (NO usar en TP2: leakage)
    "precio_suelo_m2", "precio_por_m2",
    # targets
    "Total transaction value", "Building : Structure", "estructura_principal",
]

faltantes = [c for c in columnas_finales if c not in df.columns]
if faltantes:
    raise KeyError(f"Faltan columnas esperadas: {faltantes}")

df_procesado = df[columnas_finales].copy()
df_procesado.to_csv(f"{PROCESSED_DIR}/tokyo_tp1_procesado.csv", index=False)

try:
    df_procesado.to_parquet(f"{PROCESSED_DIR}/tokyo_tp1_procesado.parquet", index=False)
except ImportError:
    print("(pyarrow/fastparquet no instalado: se omite el .parquet, el .csv ya quedó guardado)")

print("Dataset procesado guardado:", df_procesado.shape)
df_procesado.head()

## 12. Próximos pasos
Queda resuelto y justificado: artefacto `‡u` (+ flags de capado), distancia a estación parseada desde
texto, nulos estructurales por `Type`, cardinalidad geográfica por top-N, antigüedades inconsistentes,
features temporales/uso, y outliers de precio/superficie.
`outputs/processed/tokyo_tp1_procesado.parquet` es el punto de partida del **TP2** (regresión sobre
`log1p(Total transaction value)`) y del **TP3** (clasificación de `Building : Structure`, con foco en el
desbalance).